<a href="https://colab.research.google.com/github/Leandro2402-bit/Analitica_de_Datos/blob/main/Tareas/Tarea_Teoria_se%C3%B1ales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================
# Estimación de parámetros de una señal cosenoidal con ruido
# x(t) = a cos(ω0 t + φ) + n(t)
# ===============================================

import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Parámetros de la señal
# -----------------------------
fs = 1000                 # Frecuencia de muestreo [Hz]
T = 1.0                   # Duración de la señal [s]
t = np.linspace(0, T, int(fs*T), endpoint=False)  # Vector de tiempo

a = 2.0                   # Amplitud real
f0 = 50                   # Frecuencia real en Hz
w0 = 2*np.pi*f0            # Frecuencia angular
phi = np.pi/4             # Fase real (45°)

# Señal limpia
s = a * np.cos(w0*t + phi)

# -----------------------------
# 2. Ruido gaussiano
# -----------------------------
np.random.seed(0)         # Para reproducibilidad
ruido = np.random.normal(0, 1, len(t))  # Media 0, varianza 1
x = s + ruido             # Señal observada con ruido

# -----------------------------
# 3. Estimación con un coseno
#    (proyección solo sobre coseno)
# -----------------------------
cos_base = np.cos(w0*t)
A1 = (2/len(t)) * np.dot(x, cos_base)   # Coeficiente estimado
s1 = A1 * cos_base                      # Reconstrucción

# -----------------------------
# 4. Estimación con cos + sen
#    (proyección en base ortogonal)
# -----------------------------
cos_base = np.cos(w0*t)
sen_base = np.sin(w0*t)

C1 = (2/len(t)) * np.dot(x, cos_base)   # Proyección cos
C2 = (2/len(t)) * np.dot(x, sen_base)   # Proyección sen

s2 = C1*cos_base + C2*sen_base          # Reconstrucción

# Amplitud y fase estimadas
a_hat = np.sqrt(C1**2 + C2**2)
phi_hat = np.arctan2(-C2, C1)

# -----------------------------
# 5. Estimación de frecuencia con FFT
# -----------------------------
Xf = np.fft.fft(x)
freqs = np.fft.fftfreq(len(t), 1/fs)
Xf_mag = np.abs(Xf)

# Tomar solo la parte positiva del espectro
pos_mask = freqs > 0
freqs_pos = freqs[pos_mask]
Xf_mag_pos = Xf_mag[pos_mask]

# Frecuencia estimada = pico espectral
f0_hat = freqs_pos[np.argmax(Xf_mag_pos)]

# -----------------------------
# 6. Gráficas
# -----------------------------
plt.figure(figsize=(12,8))

# Señal limpia y observada
plt.subplot(3,1,1)
plt.plot(t, s, label="Señal limpia (s)", color="green")
plt.plot(t, x, label="Señal con ruido (x)", color="gray", alpha=0.6)
plt.legend()
plt.title("Señal original y señal observada con ruido")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")

# Estimaciones con 1 coseno y 2 cosenos
plt.subplot(3,1,2)
plt.plot(t, x, label="Señal con ruido", color="gray", alpha=0.5)
plt.plot(t, s1, label="Estimación (1 coseno)", color="red")
plt.plot(t, s2, label="Estimación (cos+sen)", color="blue")
plt.legend()
plt.title("Comparación de estimaciones")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")

# Espectro FFT
plt.subplot(3,1,3)
plt.plot(freqs_pos, Xf_mag_pos, color="purple")
plt.axvline(f0, color="green", linestyle="--", label=f"f0 real = {f0} Hz")
plt.axvline(f0_hat, color="red", linestyle="--", label=f"f0 estimada = {f0_hat:.2f} Hz")
plt.legend()
plt.title("Estimación de la frecuencia con FFT")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud")

plt.tight_layout()
plt.show()

# -----------------------------
# 7. Resultados numéricos
# -----------------------------
print("Resultados de la estimación:")
print(f"Amplitud real = {a}, Amplitud estimada = {a_hat:.3f}")
print(f"Fase real = {phi:.3f} rad, Fase estimada = {phi_hat:.3f} rad")
print(f"Frecuencia real = {f0} Hz, Frecuencia estimada = {f0_hat:.2f} Hz")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.fft import fft, fftfreq

# Parámetros reales de la señal (valores conocidos para comparar)
a_real = 2.0      # Amplitud real
f0_real = 5.0     # Frecuencia en Hz
w0_real = 2*np.pi*f0_real  # Frecuencia angular real
phi_real = np.pi/4  # Fase real
noise_level = 0.3   # Nivel de ruido

# Generar señal con ruido
t = np.linspace(0, 2, 1000)  # 2 segundos, 1000 puntos
signal_clean = a_real * np.cos(w0_real * t + phi_real)
noise = np.random.normal(0, noise_level, len(t))
x_t = signal_clean + noise  # Señal con ruido

print("=== PARÁMETROS REALES ===")
print(f"Amplitud: {a_real}")
print(f"Frecuencia: {f0_real} Hz")
print(f"Fase: {phi_real:.3f} rad")

# MÉTODO 1: Un coseno directo
def cosine_model(t, a, w0, phi):
    return a * np.cos(w0 * t + phi)

# Estimación inicial
popt1, _ = curve_fit(cosine_model, t, x_t, p0=[2, 30, 0])
a_est1, w0_est1, phi_est1 = popt1

# MÉTODO 2: Dos cosenos (cos + sen)
def two_cosine_model(t, A, B, w0):
    return A * np.cos(w0 * t) + B * np.sin(w0 * t)

popt2, _ = curve_fit(two_cosine_model, t, x_t, p0=[1, 1, 30])
A_est, B_est, w0_est2 = popt2
# Convertir a amplitud y fase
a_est2 = np.sqrt(A_est**2 + B_est**2)
phi_est2 = np.arctan2(-B_est, A_est)

# MÉTODO 3: Frecuencia centralizada (usando FFT)
# Encontrar frecuencia dominante con FFT
fft_signal = fft(x_t)
freqs = fftfreq(len(t), t[1]-t[0])
# Encontrar pico de frecuencia
idx_max = np.argmax(np.abs(fft_signal[1:len(freqs)//2])) + 1
f0_est3 = abs(freqs[idx_max])
w0_est3 = 2*np.pi*f0_est3

# Estimar amplitud y fase con frecuencia fija
def cosine_fixed_freq(t, a, phi):
    return a * np.cos(w0_est3 * t + phi)

popt3, _ = curve_fit(cosine_fixed_freq, t, x_t, p0=[2, 0])
a_est3, phi_est3 = popt3

# Calcular errores (incertidumbre)
def calculate_errors():
    errors = {}

    # Método 1
    errors['M1_amp'] = abs(a_est1 - a_real)
    errors['M1_freq'] = abs(w0_est1/(2*np.pi) - f0_real)
    errors['M1_phase'] = abs(phi_est1 - phi_real)

    # Método 2
    errors['M2_amp'] = abs(a_est2 - a_real)
    errors['M2_freq'] = abs(w0_est2/(2*np.pi) - f0_real)
    errors['M2_phase'] = abs(phi_est2 - phi_real)

    # Método 3
    errors['M3_amp'] = abs(a_est3 - a_real)
    errors['M3_freq'] = abs(f0_est3 - f0_real)
    errors['M3_phase'] = abs(phi_est3 - phi_real)

    return errors

errors = calculate_errors()

# Mostrar resultados
print("\n=== ESTIMACIONES ===")
print("Método 1 (1 coseno):")
print(f"  Amplitud: {a_est1:.3f} (error: {errors['M1_amp']:.3f})")
print(f"  Frecuencia: {w0_est1/(2*np.pi):.3f} Hz (error: {errors['M1_freq']:.3f})")
print(f"  Fase: {phi_est1:.3f} rad (error: {errors['M1_phase']:.3f})")

print("\nMétodo 2 (2 cosenos):")
print(f"  Amplitud: {a_est2:.3f} (error: {errors['M2_amp']:.3f})")
print(f"  Frecuencia: {w0_est2/(2*np.pi):.3f} Hz (error: {errors['M2_freq']:.3f})")
print(f"  Fase: {phi_est2:.3f} rad (error: {errors['M2_phase']:.3f})")

print("\nMétodo 3 (freq. centralizada):")
print(f"  Amplitud: {a_est3:.3f} (error: {errors['M3_amp']:.3f})")
print(f"  Frecuencia: {f0_est3:.3f} Hz (error: {errors['M3_freq']:.3f})")
print(f"  Fase: {phi_est3:.3f} rad (error: {errors['M3_phase']:.3f})")

# Generar señales estimadas para graficar
x_est1 = cosine_model(t, a_est1, w0_est1, phi_est1)
x_est2 = two_cosine_model(t, A_est, B_est, w0_est2)
x_est3 = cosine_fixed_freq(t, a_est3, phi_est3)

# GRÁFICAS
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))

# Gráfica 1: Señales originales y con ruido
ax1.plot(t[:200], signal_clean[:200], 'g-', linewidth=2, label='Señal real')
ax1.plot(t[:200], x_t[:200], 'k-', alpha=0.7, label='Señal + ruido')
ax1.set_title('Señal Original vs Señal con Ruido')
ax1.set_xlabel('Tiempo (s)')
ax1.set_ylabel('Amplitud')
ax1.legend()
ax1.grid(True)

# Gráfica 2: Comparación de estimaciones
ax2.plot(t[:200], x_t[:200], 'k-', alpha=0.5, label='Señal + ruido')
ax2.plot(t[:200], x_est1[:200], 'r-', linewidth=2, label='Método 1 (1 cos)')
ax2.plot(t[:200], x_est2[:200], 'b--', linewidth=2, label='Método 2 (2 cos)')
ax2.plot(t[:200], x_est3[:200], 'm:', linewidth=2, label='Método 3 (FFT)')
ax2.set_title('Comparación de Estimaciones')
ax2.set_xlabel('Tiempo (s)')
ax2.set_ylabel('Amplitud')
ax2.legend()
ax2.grid(True)

# Gráfica 3: Errores por componente
components = ['Amplitud', 'Frecuencia', 'Fase']
method1_errors = [errors['M1_amp'], errors['M1_freq'], errors['M1_phase']]
method2_errors = [errors['M2_amp'], errors['M2_freq'], errors['M2_phase']]
method3_errors = [errors['M3_amp'], errors['M3_freq'], errors['M3_phase']]

x_pos = np.arange(len(components))
width = 0.25

ax3.bar(x_pos - width, method1_errors, width, label='Método 1', color='red', alpha=0.7)
ax3.bar(x_pos, method2_errors, width, label='Método 2', color='blue', alpha=0.7)
ax3.bar(x_pos + width, method3_errors, width, label='Método 3', color='magenta', alpha=0.7)

ax3.set_title('Incertidumbre por Componente')
ax3.set_xlabel('Componentes')
ax3.set_ylabel('Error Absoluto')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(components)
ax3.legend()
ax3.grid(True, alpha=0.3)

# Gráfica 4: Espectro de frecuencia
freqs_plot = freqs[:len(freqs)//2]
fft_plot = np.abs(fft_signal)[:len(freqs)//2]
ax4.plot(freqs_plot, fft_plot, 'orange', linewidth=2)
ax4.axvline(f0_real, color='g', linestyle='--', label=f'Freq real: {f0_real} Hz')
ax4.axvline(f0_est3, color='m', linestyle=':', label=f'Freq estimada: {f0_est3:.2f} Hz')
ax4.set_title('Espectro de Frecuencia (FFT)')
ax4.set_xlabel('Frecuencia (Hz)')
ax4.set_ylabel('Magnitud')
ax4.set_xlim(0, 20)
ax4.legend()
ax4.grid(True)

plt.tight_layout()
plt.show()

# Análisis de incertidumbre
print("\n=== ANÁLISIS DE INCERTIDUMBRE ===")
all_errors = [method1_errors, method2_errors, method3_errors]
method_names = ['Método 1', 'Método 2', 'Método 3']

for i, component in enumerate(components):
    component_errors = [errors[i] for errors in all_errors]
    worst_method = np.argmax(component_errors)
    print(f"{component}: Mayor incertidumbre en {method_names[worst_method]} (error: {max(component_errors):.3f})")